# `lga` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `lga`.

            ## Relationships selected in advance

            - `region` — LGA maps deterministically to region in the supplied data.
- `ward` — LGA context disambiguates reused ward names.
- `region_code` — Code relationships expose small administrative anomalies.
- `district_code` — Named and coded district representations overlap.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'lga'
feature_metadata = {'order': 15, 'name': 'lga', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain with explicit unseen handling', 'finding': 'All 125 training levels are covered in test and each maps to one region.', 'decision': 'Retain and compare with the coarser region representation.', 'risk': 'Strong geographic target differences require grouped robustness checks.', 'related': [{'feature': 'region', 'reason': 'LGA maps deterministically to region in the supplied data.'}, {'feature': 'ward', 'reason': 'LGA context disambiguates reused ward names.'}, {'feature': 'region_code', 'reason': 'Code relationships expose small administrative anomalies.'}, {'feature': 'district_code', 'reason': 'Named and coded district representations overlap.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for lga.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,region,LGA maps deterministically to region in the su...
1,ward,LGA context disambiguates reused ward names.
2,region_code,Code relationships expose small administrative...
3,district_code,Named and coded district representations overlap.


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,lga,region,bias-corrected Cramer's V,0.9991,59400,125,21,100.00,31.79,LGA maps deterministically to region in the su...
1,lga,ward,bias-corrected Cramer's V,0.9637,59400,125,2092,15.04,97.62,LGA context disambiguates reused ward names.
2,lga,region_code,bias-corrected Cramer's V,0.9719,59400,125,27,99.21,34.67,Code relationships expose small administrative...
3,lga,district_code,bias-corrected Cramer's V,0.8565,59400,125,20,97.71,20.15,Named and coded district representations overlap.


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `lga`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **retain with explicit unseen handling**.
